In [12]:
import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment

In [42]:
BASE_URL = "https://www.sports-reference.com"


import requests
import pandas as pd
from bs4 import BeautifulSoup, Comment
from io import StringIO

def get_team_stats_from_csv(year):
    url = f"https://www.sports-reference.com/cbb/seasons/men/{year}-school-stats.html"
    #advanced-school-stats.html
    res = requests.get(url)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "html.parser")

    # Prefer the raw HTML table when available, otherwise fall back to commented CSV text
    table = soup.find("table", id="basic_school_stats")
    if table is not None:
        df = pd.read_html(str(table))[0]
        df.columns = [col[1] if isinstance(col, tuple) else col for col in df.columns]
        df["School"] = df["School"].astype(str).str.replace("\u00a0", " ", regex=False).str.strip()
        df = df[~df["School"].isin(["School", "Overall"])]
    else:
        comments = soup.find_all(string=lambda t: isinstance(t, Comment))
        csv_text = None
        for c in comments:
            if "csv_basic_school_stats" in c or "csv_adv_school_stats" in c:
                csv_text = c
                break

        if csv_text is None:
            raise ValueError("basic_school_stats table and CSV not found in page source")

        df = pd.read_csv(StringIO(csv_text))
        df["School"] = df["School"].astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

    # Clean up
    df = df.rename(columns={
        "School": "team",
        "Conf": "conference",
        "W": "wins",
        "L": "losses",
        "W-L%": "win_pct",
        "PS/G": "points_for",
        "PA/G": "points_against"
    })

    # Keep only NCAA tournament teams and remove rank column
    df = df[df["team"].str.endswith("NCAA")].copy()
    df["team"] = df["team"].str.replace(r"\s*NCAA$", "", regex=True).str.strip()
    df = df.drop(columns=[col for col in ["Rk", "Rank"] if col in df.columns])

    # df["wins"] = pd.to_numeric(df["wins"], errors="coerce")
    # df["losses"] = pd.to_numeric(df["losses"], errors="coerce")
    # df["win_pct"] = pd.to_numeric(df["win_pct"], errors="coerce")
    # df["points_for"] = pd.to_numeric(df["points_for"], errors="coerce")
    # df["points_against"] = pd.to_numeric(df["points_against"], errors="coerce")
    # df = df.dropna(subset=["wins", "losses", "win_pct", "points_for", "points_against"])
    # df[["wins", "losses"]] = df[["wins", "losses"]].astype(int)

    # df["avg_margin"] = df["points_for"] - df["points_against"]
    df["year"] = year

    return df


In [39]:
def get_tournament_games(year):
    url = f"{BASE_URL}/cbb/postseason/{year}-ncaa.html"
    
    res = requests.get(url)
    res.raise_for_status()
    
    soup = BeautifulSoup(res.text, "html.parser")
    
    games = []
    
    # Tournament games are organized in round containers.
    for round_div in soup.find_all("div", class_="round"):
        for game_div in round_div.find_all("div", recursive=False):
            # Each game div contains two team divs and a location box.
            team_divs = [d for d in game_div.find_all("div", recursive=False) if d.find("a")]
            if len(team_divs) < 2:
                continue

            # Determine winner and loser.
            if "winner" in (team_divs[0].get("class") or []):
                winner_div, loser_div = team_divs[0], team_divs[1]
            elif "winner" in (team_divs[1].get("class") or []):
                winner_div, loser_div = team_divs[1], team_divs[0]
            else:
                # Fallback: compare the final score anchor text.
                def parse_score(div):
                    scores = [a.text.strip() for a in div.find_all("a") if a.text.strip().isdigit()]
                    return int(scores[-1]) if scores else None
                score0 = parse_score(team_divs[0])
                score1 = parse_score(team_divs[1])
                if score0 is None or score1 is None:
                    continue
                if score0 > score1:
                    winner_div, loser_div = team_divs[0], team_divs[1]
                else:
                    winner_div, loser_div = team_divs[1], team_divs[0]

            def extract_team_name(div):
                link = div.find_all("a")
                return link[0].text.strip() if link else None

            def extract_seed(div):
                span = div.find("span")
                if span is None:
                    return None
                text = span.text.strip()
                return int(text) if text.isdigit() else None

            winner = extract_team_name(winner_div)
            loser = extract_team_name(loser_div)
            winner_seed = extract_seed(winner_div)
            loser_seed = extract_seed(loser_div)

            games.append({
                "season": year,
                "winner": winner,
                "winner_seed": winner_seed,
                "loser": loser,
                "loser_seed": loser_seed
            })
    
    return pd.DataFrame(games)

In [5]:
import re

def clean_team_name(name):
    name = re.sub(r"\*", "", name)          # remove *
    name = re.sub(r"\(\d+\)", "", name)     # remove seeds like (1)
    return name.strip()

# df["team"] = df["team"].apply(clean_team_name)
# games["winner"] = games["winner"].apply(clean_team_name)
# games["loser"] = games["loser"].apply(clean_team_name)

In [43]:
def build_dataset(start_year, end_year):
    all_teams = []
    all_games = []
    
    for year in range(start_year, end_year + 1):
        print(f"Scraping {year}...")
        
        teams = get_team_stats_from_csv(year)
        games = get_tournament_games(year)
        
        all_teams.append(teams)
        all_games.append(games)
    
    teams_df = pd.concat(all_teams, ignore_index=True)
    games_df = pd.concat(all_games, ignore_index=True)
    
    return teams_df, games_df

In [ ]:
teams_df, games_df = build_dataset(2015, 2025)

teams_df.to_csv("teams.csv", index=False)
games_df.to_csv("tournament_games.csv", index=False)